# Implementasi Content-based Filtering

**Nama:**
1. Fawwaz Rif'at Revista
2. Khalilullah Al Faath

**Mata Kuliah:** Sistem Temu Balik Informasi

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import dataset

In [ ]:
# Set the path to the file you'd like to load
file_path = "dataset.csv"

# Load the latest version
df_spotify = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "maharshipandya/-spotify-tracks-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# print("First 5 records:", df.head())

/tmp/ipykernel_18607/3428528313.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_spotify = kagglehub.load_dataset(


Using Colab cache for faster access to the '-spotify-tracks-dataset' dataset.


In [ ]:
df_spotify.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
df_spotify.shape

(114000, 21)

In [ ]:
df_spotify.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [ ]:
# drop missing value
df_spotify = df_spotify.dropna(subset=['artists', 'album_name', 'track_name'])

# drop duplicates
df_spotify = df_spotify.drop_duplicates(subset=['track_id'], keep='first')

# drop unnecessary column
df_spotify = df_spotify.drop(columns=['Unnamed: 0'])

In [ ]:
text_cols = ["artists", "track_name", "track_genre"]
audio_cols = ["energy", "danceability", "loudness", "acousticness", "valence", "tempo"]

In [ ]:
df_spotify[text_cols + audio_cols]

,artists,track_name,track_genre,energy,danceability,loudness,acousticness,valence,tempo
0,Gen Hoshino,Comedy,acoustic,0.4610,0.676,-6.746,0.0322,0.7150,87.917
1,Ben Woodward,Ghost - Acoustic,acoustic,0.1660,0.420,-17.235,0.9240,0.2670,77.489
2,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,0.3590,0.438,-9.734,0.2100,0.1200,76.332
3,Kina Grannis,Can't Help Falling In Love,acoustic,0.0596,0.266,-18.515,0.9050,0.1430,181.740
4,Chord Overstreet,Hold On,acoustic,0.4430,0.618,-9.681,0.4690,0.1670,119.949
...,...,...,...,...,...,...,...,...,...
113995,Rainy Lullaby,Sleep My Little Boy,world-music,0.2350,0.172,-16.393,0.6400,0.0339,125.995
113996,Rainy Lullaby,Water Into Light,world-music,0.1170,0.174,-18.318,0.9940,0.0350,85.239
113997,Cesária Evora,Miss Perfumado,world-music,0.3290,0.629,-10.895,0.8670,0.7430,132.378
113998,Michael W. Smith,Friends,world-music,0.5060,0.587,-10.889,0.3810,0.4130,135.960


In [ ]:
# genre unik untuk setiap kombinasi Lagu + Artis
genre_counts = df_spotify.groupby(['artists', 'track_name'])['track_genre'].nunique()

# filter lagu-lagu yang memiliki lebih dari 1 genre
multi_genre_songs = genre_counts[genre_counts > 1].sort_values(ascending=False)

print(f"Ada {len(multi_genre_songs)} lagu yang memiliki lebih dari 1 genre.")
print("\nTop 5 lagu dengan genre terbanyak:\n===============================================")
print(multi_genre_songs.head())

Ada 526 lagu yang memiliki lebih dari 1 genre.

Top 5 lagu dengan genre terbanyak:
artists                            track_name         
Weezer                             Records                3
Tiësto                             The Business           3
Wolfmother                         Woman                  3
Wisin & Yandel;Chris Brown;T-Pain  Algo Me Gusta De Ti    3
Tiësto;Charli XCX                  Hot in It              3
Name: track_genre, dtype: int64


In [ ]:
df_spotify.columns

Index(['track_id', 'artists', 'album_name', 'track_name', 'popularity',
       'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre'],
      dtype='object')

## Text Preprocessing

In [ ]:
# lowercase
df_spotify['artists_clean'] = df_spotify['artists'].str.lower()

# hapus karakter spesial
df_spotify['artists_clean'] = df_spotify['artists_clean'].str.replace(r'[^\w\s&]', '', regex=True)

# hapus spasi
df_spotify['artists_clean'] = df_spotify['artists_clean'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
# lowercase
df_spotify['track_name_clean'] = df_spotify['track_name'].str.lower()

# Hapus version info (e.g., "- remastered", "- live", "- radio edit")
patterns_to_remove = [
    r'\s*-\s*remaster.*',
    r'\s*-\s*live.*',
    r'\s*-\s*radio.*',
    r'\s*-\s*acoustic.*',
    r'\s*\(.*remaster.*\)',
    r'\s*\(.*live.*\)',
    r'\s*\(.*radio.*\)',
    r'\s*\(.*acoustic.*\)'
]
for pattern in patterns_to_remove:
    df_spotify['track_name_clean'] = df_spotify['track_name_clean'].str.replace(pattern, '', regex=True, case=False)

# hapus karakter spesial
df_spotify['track_name_clean'] = df_spotify['track_name_clean'].str.replace(r'[^\w\s]', '', regex=True)

# hapus spasi
df_spotify['track_name_clean'] = df_spotify['track_name_clean'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
# hasil setelah preprocessing
df_spotify[["artists", "artists_clean", "track_name", "track_name_clean"]]

,artists,artists_clean,track_name,track_name_clean
0,Gen Hoshino,gen hoshino,Comedy,comedy
1,Ben Woodward,ben woodward,Ghost - Acoustic,ghost
2,Ingrid Michaelson;ZAYN,ingrid michaelsonzayn,To Begin Again,to begin again
3,Kina Grannis,kina grannis,Can't Help Falling In Love,cant help falling in love
4,Chord Overstreet,chord overstreet,Hold On,hold on
...,...,...,...,...
113995,Rainy Lullaby,rainy lullaby,Sleep My Little Boy,sleep my little boy
113996,Rainy Lullaby,rainy lullaby,Water Into Light,water into light
113997,Cesária Evora,cesária evora,Miss Perfumado,miss perfumado
113998,Michael W. Smith,michael w smith,Friends,friends


In [ ]:
df_spotify.columns

Index(['track_id', 'artists', 'album_name', 'track_name', 'popularity',
       'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'track_genre', 'artists_clean',
       'track_name_clean'],
      dtype='object')

In [ ]:
print(f"Jumlah baris awal: {len(df_spotify)}")

agg_rules = {
    # gabung untuk semua
    'track_genre': lambda x: ', '.join(x.dropna().unique()),
    'track_id': 'first'
}


# nilai rata-rata
for col in audio_cols:
    if col in df_spotify.columns:
        agg_rules[col] = 'mean'

df_spotify = df_spotify.groupby(['artists_clean', 'track_name_clean'], as_index=False).agg(agg_rules)

print(f"Jumlah baris setelah dibersihkan: {len(df_spotify)}")

Jumlah baris awal: 89740
Jumlah baris setelah dibersihkan: 80687


In [ ]:
df_spotify.columns

Index(['artists_clean', 'track_name_clean', 'track_genre', 'track_id',
       'energy', 'danceability', 'loudness', 'acousticness', 'valence',
       'tempo'],
      dtype='object')

In [ ]:
# hasil gabung genre
df_spotify[["artists_clean", "track_name_clean", "track_genre"]]

,artists_clean,track_name_clean,track_genre
0,& the mysterians,96 tears,garage
1,&merampaadam portsofie royer,discoteca,deep-house
2,04 limited sazabys,horizon,j-rock
3,1 trait danger,back up hes the man,comedy
4,10 years,dont fight it,grunge
...,...,...,...
80682,黃妃,溫暖的所在,mandopop
80683,黃小琥,沒那麽簡單,mandopop
80684,黃敏華,堤岸,cantopop
80685,龍藏ryuzo,ゲゲゲの鬼太郎 instrumental,guitar


In [ ]:
# genre unik untuk setiap kombinasi Lagu + Artis
genre_counts = df_spotify.groupby(['artists_clean', 'track_name_clean'])['track_genre'].nunique()

# filter lagu-lagu yang memiliki lebih dari 1 genre
multi_genre_songs = genre_counts[genre_counts > 1].sort_values(ascending=False)

print(f"Ada {len(multi_genre_songs)} lagu yang memiliki lebih dari 1 genre.")
print("\nTop 5 lagu dengan genre terbanyak:")
print(multi_genre_songs.head())

Ada 0 lagu yang memiliki lebih dari 1 genre.

Top 5 lagu dengan genre terbanyak:
Series([], Name: track_genre, dtype: int64)


## TF-IDF vectorization (Judul + Artis)

In [ ]:
df_spotify

,artists_clean,track_name_clean,track_genre,track_id,energy,danceability,loudness,acousticness,valence,tempo
0,& the mysterians,96 tears,garage,53xKlevoXT2OIdDQx3KDl1,0.562,0.650,-5.965,0.062300,0.8800,123.674
1,&merampaadam portsofie royer,discoteca,deep-house,0ENV8cY0bwun9qSQkh195f,0.485,0.772,-14.388,0.000391,0.1560,120.000
2,04 limited sazabys,horizon,j-rock,2QWyALcWJChOa4tKXym9z6,0.994,0.420,-2.576,0.000807,0.0582,112.643
3,1 trait danger,back up hes the man,comedy,6KxbHBskjC5Kdc5MPYSsOQ,0.679,0.693,-5.452,0.072700,0.6930,159.900
4,10 years,dont fight it,grunge,2EPLbM7W9OXSYiyEdSC34y,0.738,0.409,-4.582,0.020400,0.3590,153.955
...,...,...,...,...,...,...,...,...,...,...
80682,黃妃,溫暖的所在,mandopop,0xCXMZlt1QfWfhtuTIFBpk,0.450,0.549,-7.176,0.461000,0.2340,90.062
80683,黃小琥,沒那麽簡單,mandopop,4xZIMRwaaBx7CZMmM6KLuh,0.431,0.334,-5.861,0.672000,0.2120,201.701
80684,黃敏華,堤岸,cantopop,1Q5d3X55pI7nBXYAdZ0g8Z,0.478,0.549,-6.186,0.783000,0.3410,125.917
80685,龍藏ryuzo,ゲゲゲの鬼太郎 instrumental,guitar,5Qn6Ys1fHlef8zgCLqCdud,0.325,0.571,-11.442,0.404000,0.3200,116.457


In [ ]:
df_spotify.columns

Index(['artists_clean', 'track_name_clean', 'track_genre', 'track_id',
       'energy', 'danceability', 'loudness', 'acousticness', 'valence',
       'tempo'],
      dtype='object')

In [ ]:
# df_spotify["combined_text"] = df_spotify['artists_clean'] + ' ' + df_spotify['track_name_clean']

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_artists = TfidfVectorizer(
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r'\b\w+\b'
)

# Buat objek vectorizer KEDUA khusus untuk Judul
tfidf_title = TfidfVectorizer(
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r'\b\w+\b'
)

artists_features = tfidf_artists.fit_transform(df_spotify['artists_clean'])
title_features = tfidf_title.fit_transform(df_spotify['track_name_clean'])

In [ ]:
artists_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 179581 stored elements and shape (80687, 42003)>

In [ ]:
artists_features[:5].toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.57735027, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [ ]:
total_elements = artists_features.shape[0] * artists_features.shape[1]
total_non_zeros = artists_features.nnz
total_zeros = total_elements - total_non_zeros
sparsity_percentage = (total_zeros / total_elements) * 100

print(f"Total elemen matriks : {total_elements:,}")
print(f"Jumlah elemen isi    : {total_non_zeros:,}")
print(f"Jumlah elemen 0      : {total_zeros:,}")
print(f"Persentase nilai 0   : {sparsity_percentage:.4f}%")

Total elemen matriks : 3,389,096,061
Jumlah elemen isi    : 179,581
Jumlah elemen 0      : 3,388,916,480
Persentase nilai 0   : 99.9947%


In [ ]:
title_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 250081 stored elements and shape (80687, 48788)>

In [ ]:
title_features[:5].toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
total_elements = title_features.shape[0] * title_features.shape[1]
total_non_zeros = title_features.nnz
total_zeros = total_elements - total_non_zeros
sparsity_percentage = (total_zeros / total_elements) * 100

print(f"Total elemen matriks : {total_elements:,}")
print(f"Jumlah elemen isi    : {total_non_zeros:,}")
print(f"Jumlah elemen 0      : {total_zeros:,}")
print(f"Persentase nilai 0   : {sparsity_percentage:.4f}%")

Total elemen matriks : 3,936,557,356
Jumlah elemen isi    : 250,081
Jumlah elemen 0      : 3,936,307,275
Persentase nilai 0   : 99.9936%


## Fitur genre

In [ ]:
genre_vec = CountVectorizer(tokenizer=lambda x: x.split(', '))
genre_features = genre_vec.fit_transform(df_spotify['track_genre'])

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [ ]:
genre_features

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 81296 stored elements and shape (80687, 113)>

In [ ]:
genre_features[:5].toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
  

In [ ]:
total_elements = genre_features.shape[0] * genre_features.shape[1]
total_non_zeros = genre_features.nnz
total_zeros = total_elements - total_non_zeros
sparsity_percentage = (total_zeros / total_elements) * 100

print(f"Total elemen matriks : {total_elements:,}")
print(f"Jumlah elemen isi    : {total_non_zeros:,}")
print(f"Jumlah elemen 0      : {total_zeros:,}")
print(f"Persentase nilai 0   : {sparsity_percentage:.4f}%")

Total elemen matriks : 9,117,631
Jumlah elemen isi    : 81,296
Jumlah elemen 0      : 9,036,335
Persentase nilai 0   : 99.1084%


## Fitur audio

In [ ]:
cols_audio_to_normalize = ['loudness', 'tempo']

In [ ]:
audio_features_raw = df_spotify[audio_cols].copy()

In [ ]:
# Cek missing value
print(audio_features_raw.isnull().sum())

# Cek distribusi
print(audio_features_raw.describe())

energy          0
danceability    0
loudness        0
acousticness    0
valence         0
tempo           0
dtype: int64
             energy  danceability      loudness  acousticness       valence  \
count  80687.000000  80687.000000  80687.000000  80687.000000  80687.000000   
mean       0.635133      0.559569     -8.595739      0.329600      0.463205   
std        0.258537      0.177726      5.308963      0.339664      0.263295   
min        0.000000      0.000000    -49.531000      0.000000      0.000000   
25%        0.456000      0.447000    -10.450250      0.016000      0.241000   
50%        0.678000      0.573000     -7.264000      0.191000      0.449000   
75%        0.856000      0.690000     -5.140000      0.628000      0.676000   
max        1.000000      0.985000      4.532000      0.996000      0.995000   

              tempo  
count  80687.000000  
mean     122.131860  
std       30.064513  
min        0.000000  
25%       99.512500  
50%      122.026000  
75%      140.

In [ ]:
# audio_scaler = MinMaxScaler()

# array_normalized = audio_scaler.fit_transform(df_spotify[cols_audio_to_normalize])

# rest_audio_features = df_spotify.drop(cols_audio_to_normalize, axis=1)

# audio_features_normalized = pd.DataFrame(
#     array_normalized,
#     columns=cols_audio_to_normalize,
#     index=df_spotify.index
# )

# audio_features_final = pd.concat([rest_audio_features, audio_features_normalized], axis=1)

# audio_features_sparse2 = csr_matrix(audio_features_final)

In [ ]:
cols_audio_to_normalize = ['loudness', 'tempo']
audio_scaler = MinMaxScaler()

# 1. LANGSUNG UBAH DI TEMPAT: Timpa kolom asli dengan hasil normalisasi
df_spotify[cols_audio_to_normalize] = audio_scaler.fit_transform(df_spotify[cols_audio_to_normalize])

# 2. SEKARANG KITA BUAT SPARSE MATRIX-NYA
# Karena df_spotify[cols_audio_to_normalize] sudah otomatis berubah menjadi angka 0-1,
# kamu tinggal mengambil daftar semua kolom audio (audio_cols) dari df_spotify untuk di-sparse.
audio_features_sparse2 = csr_matrix(df_spotify[audio_cols])

In [ ]:
audio_features_2_cols = df_spotify[['energy', 'valence']]

# Ubah menjadi matriks sparse
audio_features_sparse1 = csr_matrix(audio_features_2_cols)

In [ ]:
df_spotify[cols_audio_to_normalize]

,loudness,tempo
0,0.805838,0.508169
1,0.650038,0.493072
2,0.868524,0.462843
3,0.815327,0.657019
4,0.831419,0.632591
...,...,...
80682,0.783438,0.370059
80683,0.807761,0.828777
80684,0.801750,0.517385
80685,0.704530,0.478514


In [ ]:
print(df_spotify.describe())

             energy  danceability      loudness  acousticness       valence  \
count  80687.000000  80687.000000  80687.000000  80687.000000  80687.000000   
mean       0.635133      0.559569      0.757177      0.329600      0.463205   
std        0.258537      0.177726      0.098200      0.339664      0.263295   
min        0.000000      0.000000      0.000000      0.000000      0.000000   
25%        0.456000      0.447000      0.722874      0.016000      0.241000   
50%        0.678000      0.573000      0.781810      0.191000      0.449000   
75%        0.856000      0.690000      0.821098      0.628000      0.676000   
max        1.000000      0.985000      1.000000      0.996000      0.995000   

              tempo  
count  80687.000000  
mean       0.501832  
std        0.123533  
min        0.000000  
25%        0.408891  
50%        0.501397  
75%        0.575656  
max        1.000000  


In [ ]:
combined_features1 = hstack([
    artists_features,
    genre_features,
    audio_features_sparse1
]).tocsr()

In [ ]:
combined_features2 = hstack([
    artists_features,
    title_features,
    genre_features,
    audio_features_sparse2
]).tocsr()

In [ ]:
combined_features2[:5].toarray()

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        6.23000000e-02, 8.80000000e-01, 5.08168565e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.91000000e-04, 1.56000000e-01, 4.93072334e-01],
       [5.77350269e-01, 0.00000000e+00, 0.00000000e+00, ...,
        8.07000000e-04, 5.82000000e-02, 4.62842891e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        7.27000000e-02, 6.93000000e-01, 6.57018885e-01],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        2.04000000e-02, 3.59000000e-01, 6.32591259e-01]])

In [ ]:
total_elements = combined_features2.shape[0] * combined_features2.shape[1]
total_non_zeros = combined_features2.nnz
total_zeros = total_elements - total_non_zeros
sparsity_percentage = (total_zeros / total_elements) * 100

print(f"Total elemen matriks : {total_elements:,}")
print(f"Jumlah elemen isi    : {total_non_zeros:,}")
print(f"Jumlah elemen 0      : {total_zeros:,}")
print(f"Persentase nilai 0   : {sparsity_percentage:.4f}%")

Total elemen matriks : 7,335,255,170
Jumlah elemen isi    : 994,593
Jumlah elemen 0      : 7,334,260,577
Persentase nilai 0   : 99.9864%


# Metode 1: User pertama kali
Input: Artis, Genre, Valence, Energy

In [ ]:
df_spotify.columns

Index(['artists_clean', 'track_name_clean', 'track_genre', 'track_id',
       'energy', 'danceability', 'loudness', 'acousticness', 'valence',
       'tempo'],
      dtype='object')

In [ ]:
def recommend_from_criteria(artist, genre, target_energy, target_valence, top_n=5):
    """
    artist: string (contoh: 'Coldplay')
    genre: string (contoh: 'pop, rock')
    target_valence: float 0.0 - 1.0 (0: sedih/depresi, 1: bahagia/ceria)
    target_energy: float 0.0 - 1.0 (0: pelan/akustik, 1: berisik/intens)
    """

    # siapkan array audio
    # energy	danceability	loudness	acousticness	valence	tempo
    # isi nilai 0.5 untuk fitur yang tidak ditargetkan (netral di tengah skala MinMax)
    audio_values = [
        target_energy,  # energy
        target_valence  # valence
    ]

    # vektorisasi Teks dan Genre (Ubah ke dimensi matriks latih)
    q_text1 = tfidf_artists.transform([artist])
    q_genre = genre_vec.transform([genre])

    # vektorisasi Audio
    q_audio = csr_matrix([audio_values])

    # gabungkan vektor
    q_vector = hstack([q_text1, q_genre, q_audio]).tocsr()

    # hitung jarak cosine similarity dengan matriks dataset
    sim_scores = cosine_similarity(q_vector, combined_features1)[0]

    # urutkan dari yang paling mirip dan ambil Top-N
    top_indices = sim_scores.argsort()[::-1][:top_n]

    # Ambil data lagunya (tambahkan kolom valence dan energy agar hasilnya bisa dievaluasi)
    hasil = df_spotify.iloc[top_indices][['track_name_clean', 'artists_clean', 'track_genre', 'valence', 'energy']].copy()
    hasil['similarity_score'] = sim_scores[top_indices]

    return hasil

print(f"Mencari lagu dari artis 'adele', genre 'british', nuansa positive (Valence: 0.1) dan calm (Energy: 0.2)")

hasil_test = recommend_from_criteria(
    artist="adele",
    genre="british",
    target_valence=0.1,  # Sedih
    target_energy=0.2,   # Pelan / Kurang Intens
    top_n=5
)

print(hasil_test)

Mencari lagu dari artis 'adele', genre 'british', nuansa positive (Valence: 0.1) dan calm (Energy: 0.2)
      track_name_clean artists_clean track_genre  valence  energy  \
932  million years ago         adele     british    0.179   0.274   
927   love in the dark         adele     british    0.152   0.341   
919         easy on me         adele     british    0.130   0.366   
936             remedy         adele     british    0.240   0.300   
923     hometown glory         adele     british    0.210   0.336   

     similarity_score  
932          0.997276  
927          0.994835  
919          0.993513  
936          0.993217  
923          0.993049  


# Metode 2:
User input lagu yang sudah disukai sebelumnya

In [ ]:
def recommend_from_seeds(seed_songs, top_n=5):
    """
    Format input seed_songs:
    [("Judul Lagu 1", "Nama Artis 1"), ("Judul Lagu 2", "Nama Artis 2")]
    """
    song_indices = []

    # 1cari index untuk setiap kombinasi lagu dan artis
    for track, artist in seed_songs:
        mask = (df_spotify['track_name_clean'].str.lower() == track.lower()) & \
               (df_spotify['artists_clean'].str.lower() == artist.lower())

        match = df_spotify[mask]

        if not match.empty:
            # Ambil index dari lagu yang cocok
            song_indices.append(match.index[0])
        else:
            print(f"Peringatan: Lagu '{track}' dari artis '{artist}' tidak ditemukan di dataset.")

    # Validasi jika tidak ada satupun lagu yang ketemu
    if not song_indices:
        return "Tidak ada satupun lagu input yang valid untuk diproses."

    found_songs_df = df_spotify.iloc[song_indices][['track_name_clean', 'artists_clean', 'track_genre']]
    print("\nLagu input yang terdeteksi di sistem:")
    print(found_songs_df.to_string(index=False))
    print("-" * 65)

    # ambil vektor dari lagu-lagu tersebut
    seed_vectors = combined_features2[song_indices]

    # buat profil user
    user_profile_vector = seed_vectors.mean(axis=0)
    user_profile_vector = np.asarray(user_profile_vector).flatten()

    # hitung Cosine Similarity
    sim_scores = cosine_similarity(user_profile_vector.reshape(1, -1), combined_features2)[0]

    # urutkan berdasarkan kemiripan tertinggi
    sorted_indices = sim_scores.argsort()[::-1]

    # ambil Top-N (pastikan indeks lagu input diabaikan)
    recommended_indices = [idx for idx in sorted_indices if idx not in song_indices][:top_n]

    hasil_rekomendasi = df_spotify.iloc[recommended_indices][['track_name_clean', 'artists_clean', 'track_genre']].copy()
    hasil_rekomendasi['similarity_score'] = sim_scores[recommended_indices]

    return hasil_rekomendasi

In [ ]:
daftar_lagu_user = [
    ("Yellow", "Coldplay"),
    ("Gravity", "John Mayer")
]

hasil = recommend_from_seeds(seed_songs=daftar_lagu_user, top_n=5)

print("\n Hasil rekomendasi:")
print(hasil)


Lagu input yang terdeteksi di sistem:
track_name_clean artists_clean       track_genre
          yellow      coldplay               pop
         gravity    john mayer singer-songwriter
-----------------------------------------------------------------

 Hasil rekomendasi:
                     track_name_clean artists_clean        track_genre  \
35042                             say    john mayer  singer-songwriter   
35038             love on the weekend    john mayer  singer-songwriter   
35041                           rosie    john mayer  singer-songwriter   
35049  waiting on the world to change    john mayer  singer-songwriter   
35028                       daughters    john mayer  singer-songwriter   

       similarity_score  
35042          0.715128  
35038          0.709013  
35041          0.707724  
35049          0.707464  
35028          0.699847  


# Pretrained Word2Vec

In [ ]:
# !pip install gensim

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# from gensim.models import Word2Vec

# model_path = "/content/drive/MyDrive/STBI/Word2Vec/wiki.id.case.model"

# wv3 = Word2Vec.load(model_path)
# print(f"Vector size: {wv3.vector_size}")

In [ ]:
# w2v_matrix = np.array([get_doc_vector(doc, wv3) for doc in df_content_only['content']])
# print(f"Ukuran w2v_matrix: {w2v_matrix.shape}")

In [ ]:
df_spotify_copy = df_spotify.copy()

In [ ]:
df_spotify_copy

,artists_clean,track_name_clean,track_genre,track_id,energy,danceability,loudness,acousticness,valence,tempo
0,& the mysterians,96 tears,garage,53xKlevoXT2OIdDQx3KDl1,0.562,0.650,0.805838,0.062300,0.8800,0.508169
1,&merampaadam portsofie royer,discoteca,deep-house,0ENV8cY0bwun9qSQkh195f,0.485,0.772,0.650038,0.000391,0.1560,0.493072
2,04 limited sazabys,horizon,j-rock,2QWyALcWJChOa4tKXym9z6,0.994,0.420,0.868524,0.000807,0.0582,0.462843
3,1 trait danger,back up hes the man,comedy,6KxbHBskjC5Kdc5MPYSsOQ,0.679,0.693,0.815327,0.072700,0.6930,0.657019
4,10 years,dont fight it,grunge,2EPLbM7W9OXSYiyEdSC34y,0.738,0.409,0.831419,0.020400,0.3590,0.632591
...,...,...,...,...,...,...,...,...,...,...
80682,黃妃,溫暖的所在,mandopop,0xCXMZlt1QfWfhtuTIFBpk,0.450,0.549,0.783438,0.461000,0.2340,0.370059
80683,黃小琥,沒那麽簡單,mandopop,4xZIMRwaaBx7CZMmM6KLuh,0.431,0.334,0.807761,0.672000,0.2120,0.828777
80684,黃敏華,堤岸,cantopop,1Q5d3X55pI7nBXYAdZ0g8Z,0.478,0.549,0.801750,0.783000,0.3410,0.517385
80685,龍藏ryuzo,ゲゲゲの鬼太郎 instrumental,guitar,5Qn6Ys1fHlef8zgCLqCdud,0.325,0.571,0.704530,0.404000,0.3200,0.478514


# Collaborative

In [ ]:
import pandas as pd
import numpy as np

# 1. Ambil semua daftar track_id unik yang BENAR-BENAR ada di dataset Spotify kamu
unique_tracks = df_spotify_copy['track_id'].unique()

# Atur random seed agar hasilnya konsisten saat dijalankan ulang
np.random.seed(42)

user_ids = []
track_ids = []
track_title = []
ratings = []

# 2. Loop untuk membuat data tiruan dari User_1 sampai User_20
for i in range(1, 21):
    user_name = f'user_{i}'

    # Anggap setiap user mendengarkan antara 15 hingga 40 lagu secara acak
    num_songs = np.random.randint(15, 41)

    # Ambil sampel track_id secara acak dari database tanpa duplikat untuk user yang sama
    sampled_tracks = np.random.choice(unique_tracks, size=num_songs, replace=False)

    # Berikan rating acak (1 sampai 5)
    # Pilihan p=[...] di bawah ini opsional, gunanya agar rating 4 dan 5 lebih sering muncul (lebih realistis)
    sampled_ratings = np.random.choice([1, 2, 3, 4, 5], size=num_songs, p=[0.1, 0.1, 0.2, 0.3, 0.3])

    # Masukkan data ke dalam list utama
    user_ids.extend([user_name] * num_songs)
    track_ids.extend(sampled_tracks)
    ratings.extend(sampled_ratings)

# 3. Satukan seluruh list menjadi sebuah DataFrame baru
df_ratings = pd.DataFrame({
    'user_id': user_ids,
    'track_id': track_ids,
    'rating': ratings
})

# 4. Cek ringkasan datanya
print(f"Total baris interaksi yang berhasil dibuat: {df_ratings.shape[0]} baris")
print("\n5 DATA TERATAS")
print(df_ratings.head())

print("\nJUMLAH LAGU YANG DIDENGAR PER USER")
print(df_ratings['user_id'].value_counts())

Total baris interaksi yang berhasil dibuat: 529 baris

5 DATA TERATAS
  user_id                track_id  rating
0  user_1  0wqqLC1IRgsgivUAmna9F4       1
1  user_1  5vodiGCX9G2V22tb0LXIdg       1
2  user_1  0Yld4eVEV6rBvpijVU2s6l       3
3  user_1  6kiEh1xj80odDVe7Sq9QNY       4
4  user_1  7d9UIIsx2HnmQicVjzBIHi       5

JUMLAH LAGU YANG DIDENGAR PER USER
user_id
user_20    38
user_10    38
user_15    36
user_8     35
user_17    33
user_18    32
user_12    30
user_3     30
user_13    28
user_6     25
user_7     25
user_19    25
user_4     24
user_11    24
user_1     21
user_16    19
user_2     18
user_9     17
user_5     16
user_14    15
Name: count, dtype: int64


In [ ]:
# 1. Ambil semua daftar track_id unik yang BENAR-BENAR ada di dataset Spotify kamu
unique_tracks = df_spotify_copy['track_id'].unique()

# Atur random seed agar hasilnya konsisten saat dijalankan ulang
np.random.seed(42)

user_ids = []
track_ids = []
ratings = []

# 2. Loop untuk membuat data tiruan dari User_1 sampai User_20
for i in range(1, 21):
    user_name = f'user_{i}'

    # Anggap setiap user mendengarkan antara 15 hingga 40 lagu secara acak
    num_songs = np.random.randint(15, 41)

    # Ambil sampel track_id secara acak dari database tanpa duplikat untuk user yang sama
    sampled_tracks = np.random.choice(unique_tracks, size=num_songs, replace=False)

    # Berikan rating acak (1 sampai 5)
    sampled_ratings = np.random.choice([1, 2, 3, 4, 5], size=num_songs, p=[0.1, 0.1, 0.2, 0.3, 0.3])

    # Masukkan data ke dalam list utama
    user_ids.extend([user_name] * num_songs)
    track_ids.extend(sampled_tracks)
    ratings.extend(sampled_ratings)

# 3. Satukan seluruh list menjadi sebuah DataFrame baru
df_ratings = pd.DataFrame({
    'user_id': user_ids,
    'track_id': track_ids,
    'rating': ratings
})

# ==================== BARIS BARU DI SINI ====================
# Ambil info judul lagu dan artis dari dataset asli, buang jika ada id yang duplikat
df_track_info = df_spotify_copy[['track_id', 'track_name_clean', 'artists_clean']].drop_duplicates(subset=['track_id'])

# Gabungkan ke df_ratings berdasarkan kecocokan 'track_id'
df_ratings = pd.merge(df_ratings, df_track_info, on='track_id', how='left')

# Atur urutan kolom agar lebih enak dibaca (Opsional)
df_ratings = df_ratings[['user_id', 'track_id', 'track_name_clean', 'artists_clean', 'rating']]
# ============================================================

# 4. Cek ringkasan datanya
print(f"Total baris interaksi yang berhasil dibuat: {df_ratings.shape[0]} baris")
print("\n--- 5 DATA TERATAS ---")
print(df_ratings.head())

print("\n--- JUMLAH LAGU YANG DIDENGAR PER USER ---")
print(df_ratings['user_id'].value_counts())

Total baris interaksi yang berhasil dibuat: 529 baris

--- 5 DATA TERATAS ---
  user_id                track_id  \
0  user_1  0wqqLC1IRgsgivUAmna9F4   
1  user_1  5vodiGCX9G2V22tb0LXIdg   
2  user_1  0Yld4eVEV6rBvpijVU2s6l   
3  user_1  6kiEh1xj80odDVe7Sq9QNY   
4  user_1  7d9UIIsx2HnmQicVjzBIHi   

                                 track_name_clean  \
0                                jamás retornarás   
1                     no newstyle buzz fuzz remix   
2                                           naina   
3  the impossible dream from the man of la mancha   
4                 cant hold me back original edit   

                                  artists_clean  rating  
0                         miguel caloraúl berón       1  
1                        the masochistbuzz fuzz       1  
2                            pritamarijit singh       3  
3  josé carrerasrobert farnon and his orchestra       4  
4                             code blacknitrouz       5  

--- JUMLAH LAGU YANG DIDENGAR P

In [ ]:
df_ratings

,user_id,track_id,track_name_clean,artists_clean,rating
0,user_1,0wqqLC1IRgsgivUAmna9F4,jamás retornarás,miguel caloraúl berón,1
1,user_1,5vodiGCX9G2V22tb0LXIdg,no newstyle buzz fuzz remix,the masochistbuzz fuzz,1
2,user_1,0Yld4eVEV6rBvpijVU2s6l,naina,pritamarijit singh,3
3,user_1,6kiEh1xj80odDVe7Sq9QNY,the impossible dream from the man of la mancha,josé carrerasrobert farnon and his orchestra,4
4,user_1,7d9UIIsx2HnmQicVjzBIHi,cant hold me back original edit,code blacknitrouz,5
...,...,...,...,...,...
524,user_20,4NzSTvQA15Js1qsj2eqUB3,brick windows,ministry,3
525,user_20,5pVFQcYx5ESJDoB9ZZPVdb,disaster,zamdane,3
526,user_20,7EsjkelQuoUlJXEw7SeVV4,black or white,michael jackson,5
527,user_20,37HBDUutFewhdAdXY3zGIe,visions,josé gonzález,5


In [ ]:
# import joblib
# joblib.dump(combined_features2, 'model.pkl')

In [ ]:
# Menggabungkan user_id dengan FITUR LAIN
fitur_audio_genre = df_spotify[[
    'track_id', 'track_genre', 'energy', 'danceability',
    'loudness', 'acousticness', 'valence', 'tempo'
]]

df_ratings_complete = pd.merge(df_ratings, fitur_audio_genre, on='track_id', how='left')
df_ratings_complete

,user_id,track_id,track_name_clean,artists_clean,rating,track_genre,energy,danceability,loudness,acousticness,valence,tempo
0,user_1,0wqqLC1IRgsgivUAmna9F4,jamás retornarás,miguel caloraúl berón,1,tango,0.304,0.700,0.779184,0.934000,0.5130,0.497440
1,user_1,5vodiGCX9G2V22tb0LXIdg,no newstyle buzz fuzz remix,the masochistbuzz fuzz,1,happy,0.727,0.455,0.702088,0.004840,0.0442,0.677444
2,user_1,0Yld4eVEV6rBvpijVU2s6l,naina,pritamarijit singh,3,indian,0.445,0.501,0.760206,0.772000,0.3880,0.329068
3,user_1,6kiEh1xj80odDVe7Sq9QNY,the impossible dream from the man of la mancha,josé carrerasrobert farnon and his orchestra,4,opera,0.294,0.280,0.643287,0.918000,0.5020,0.388385
4,user_1,7d9UIIsx2HnmQicVjzBIHi,cant hold me back original edit,code blacknitrouz,5,hardstyle,0.893,0.328,0.844219,0.015100,0.2430,0.613727
...,...,...,...,...,...,...,...,...,...,...,...,...
524,user_20,4NzSTvQA15Js1qsj2eqUB3,brick windows,ministry,3,industrial,0.971,0.598,0.825629,0.000007,0.6640,0.476780
525,user_20,5pVFQcYx5ESJDoB9ZZPVdb,disaster,zamdane,3,french,0.468,0.641,0.760650,0.759000,0.4860,0.501779
526,user_20,7EsjkelQuoUlJXEw7SeVV4,black or white,michael jackson,5,soul,0.901,0.518,0.846827,0.174000,0.8730,0.472634
527,user_20,37HBDUutFewhdAdXY3zGIe,visions,josé gonzález,5,goth,0.245,0.496,0.697057,0.868000,0.2130,0.403563


In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import vstack, csr_matrix

# Kamus Pemetaan (Mapping)
track_id_to_index = pd.Series(df_spotify.index, index=df_spotify['track_id']).to_dict()

# Ambil daftar semua user_id yang ada di df_ratings (Total 20 user)
all_users = df_ratings['user_id'].unique()

# Siapkan list kosong untuk menampung hasil akhir
user_profiles_list = []
valid_user_ids = []

# Looping untuk membuat vektor tiap-tiap user
for id_user in all_users:
    # Filter riwayat lagu dan rating khusus untuk user
    data_user = df_ratings[df_ratings['user_id'] == id_user]

    lagu_user = data_user['track_id'].tolist()
    rating_user = data_user['rating'].values

    # Cari indeks baris lagu-lagu tersebut di dalam combined_features2
    indeks_matriks = []
    rating_valid = []

    for lagu, rating in zip(lagu_user, rating_user):
        # Cek keberadaan lagu di df_spotify
        if lagu in track_id_to_index:
            indeks_matriks.append(track_id_to_index[lagu])
            rating_valid.append(rating)

    # Jika user ini mendengarkan lagu yang ternyata tidak ada di matriks, lewati
    if len(indeks_matriks) == 0:
        continue

    rating_valid = np.array(rating_valid)

    # Ekstrak matriks fitur khusus untuk lagu-lagu yang didengar user target
    fitur_lagu_pilihan = combined_features2[indeks_matriks]

    # Hitung Rata-rata Terbobot (Weighted Average)
    user_profile_vector = fitur_lagu_pilihan.T.dot(rating_valid) / np.sum(rating_valid)

    # Hasil rumusnya adalah array 1D, kita ubah kembali jadi matriks sparse 1 baris
    user_profile_sparse = csr_matrix(user_profile_vector)

    # Simpan vektor user yang sudah jadi ke dalam list
    user_profiles_list.append(user_profile_sparse)
    valid_user_ids.append(id_user)

# Gabungkan semua vektor user tersebut menjadi satu matriks raksasa
user_profiles_matrix = vstack(user_profiles_list)

# Cek hasil akhirnya
print(f"Ukuran matriks profil user: {user_profiles_matrix.shape}")

Ukuran matriks profil user: (20, 90910)


In [ ]:
user_profiles_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3262 stored elements and shape (20, 90910)>

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Komputasi Cosine Similarity Antar Pengguna (User-to-User)
user_user_similarities = cosine_similarity(user_profiles_matrix)

print(f"Dimensi matriks kemiripan antar user: {user_user_similarities.shape}")
print("-" * 50)

# Parameter membatasi output similaritas
top_n = 3

# Iterasi untuk Seluruh Pengguna
for i, user_target_id in enumerate(valid_user_ids):
    print(f"Top {top_n} User Termirip untuk User ID: {user_target_id}")

    # Ekstraksi array 1-Dimensi yang berisi skor kemiripan target user dengan semua user lain
    skor_target = user_user_similarities[i]

    # Mengurutkan indeks dari skor kemiripan tertinggi ke terendah
    indeks_urut = np.argsort(skor_target)[::-1]

    # Inisialisasi variabel penghitung untuk membatasi output
    jumlah_ditampilkan = 0

    # 3. Iterasi Internal untuk menampilkan hasil perbandingan
    for idx in indeks_urut:
        # Logika eliminasi: Lewati komputasi jika user dibandingkan dengan dirinya sendiri
        if idx == i:
            continue

        user_pembanding_id = valid_user_ids[idx]
        skor_kemiripan = skor_target[idx]

        # Tampilkan hasil ke layar
        print(f"User ID: {user_pembanding_id} | Similarity: {skor_kemiripan:.4f}")

        # Increment penghitung dan hentikan perulangan jika batas top_n tercapai
        jumlah_ditampilkan += 1
        if jumlah_ditampilkan >= top_n:
            break

    # Cetak baris kosong sebagai pemisah antar user target sebelum melanjutkan iterasi global
    print("")

Dimensi matriks kemiripan antar user: (20, 20)
--------------------------------------------------
Top 3 User Termirip untuk User ID: user_1
User ID: user_8 | Similarity: 0.9279
User ID: user_10 | Similarity: 0.9265
User ID: user_20 | Similarity: 0.9257

Top 3 User Termirip untuk User ID: user_2
User ID: user_15 | Similarity: 0.9280
User ID: user_3 | Similarity: 0.9220
User ID: user_18 | Similarity: 0.9218

Top 3 User Termirip untuk User ID: user_3
User ID: user_15 | Similarity: 0.9476
User ID: user_8 | Similarity: 0.9473
User ID: user_10 | Similarity: 0.9436

Top 3 User Termirip untuk User ID: user_4
User ID: user_15 | Similarity: 0.9475
User ID: user_17 | Similarity: 0.9420
User ID: user_3 | Similarity: 0.9417

Top 3 User Termirip untuk User ID: user_5
User ID: user_15 | Similarity: 0.9288
User ID: user_20 | Similarity: 0.9267
User ID: user_8 | Similarity: 0.9252

Top 3 User Termirip untuk User ID: user_6
User ID: user_15 | Similarity: 0.9492
User ID: user_8 | Similarity: 0.9408
User 

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Cosine Similarity Antar Pengguna (User-to-User)
user_user_similarities = cosine_similarity(user_profiles_matrix)

print(f"Dimensi matriks kemiripan antar user: {user_user_similarities.shape}")
print("-" * 50)

# Parameter pembatasan output
top_n = 3       # Jumlah user termirip yang dicari
top_k_songs = 5 # Jumlah lagu yang ingin direkomendasikan

# Iterasi Global untuk Seluruh Pengguna
for i, user_target_id in enumerate(valid_user_ids):
    print(f"Rekomendasi untuk User ID: {user_target_id}")

    skor_target = user_user_similarities[i]
    indeks_urut = np.argsort(skor_target)[::-1]

    # Ambil lagu yang sudah didengar target agar tidak direkomendasikan ulang
    riwayat_target = set(df_ratings[df_ratings['user_id'] == user_target_id]['track_id'])

    # Dictionary untuk menampung skor kandidat lagu
    kandidat_skor = {}
    kandidat_sim_total = {}
    jumlah_ditampilkan = 0

    print(f"Top {top_n} user termirip:")

    # Iterasi Internal untuk mengekstrak data dari user pembanding
    for idx in indeks_urut:
        if idx == i:
            continue # Lewati diri sendiri

        user_pembanding_id = valid_user_ids[idx]
        skor_kemiripan = skor_target[idx]

        print(f" -user ID: {user_pembanding_id} | Similarity: {skor_kemiripan:.4f}")

        # Rekomendasi: Ekstrak lagu & rating dari user pembanding
        data_pembanding = df_ratings[df_ratings['user_id'] == user_pembanding_id]
        lagu_pembanding = data_pembanding['track_id'].values
        rating_pembanding = data_pembanding['rating'].values

        # Kalkulasi skor untuk tiap lagu
        for lagu, rating in zip(lagu_pembanding, rating_pembanding):
            # Jika target sudah pernah dengar lagu ini, lewati (eliminasi)
            if lagu in riwayat_target:
                continue

            # Menghitung bobot (Similarity x Rating)
            bobot_lagu = skor_kemiripan * rating

            # Akumulasikan ke dalam dictionary
            if lagu in kandidat_skor:
                kandidat_skor[lagu] += bobot_lagu
                kandidat_sim_total[lagu] += skor_kemiripan
            else:
                kandidat_skor[lagu] = bobot_lagu
                kandidat_sim_total[lagu] = skor_kemiripan

        jumlah_ditampilkan += 1
        if jumlah_ditampilkan >= top_n:
            break

    # Ranking lagu
    if not kandidat_skor:
        print(" Tidak ada lagu baru untuk direkomendasikan.\n")
        continue

    # --- PERBAIKAN 1: Inisialisasi dictionary kosong terlebih dahulu ---
    kandidat_skor_normalisasi = {}
    for lagu, total_bobot in kandidat_skor.items():
        total_sim = kandidat_sim_total[lagu]
        kandidat_skor_normalisasi[lagu] = total_bobot / total_sim if total_sim > 0 else 0

    # --- PERBAIKAN 2: Mengurutkan menggunakan variabel penampung baru yang sudah dibagi ---
    lagu_terurut = sorted(kandidat_skor_normalisasi.items(), key=lambda x: x[1], reverse=True)
    rekomendasi_final = lagu_terurut[:top_k_songs]

    print("\nTop Rekomendasi Lagu:")
    for rank, (track_id, skor) in enumerate(rekomendasi_final, 1):
        judul_lagu = df_spotify[df_spotify['track_id'] == track_id]['track_name_clean'].values[0]
        print(f" {rank}. {judul_lagu} | Track ID: {track_id} | Skor: {skor:.4f}")

    print("\n" + "="*50 + "\n")

Dimensi matriks kemiripan antar user: (20, 20)
--------------------------------------------------
Rekomendasi untuk User ID: user_1
Top 3 user termirip:
 -user ID: user_8 | Similarity: 0.9279
 -user ID: user_10 | Similarity: 0.9265
 -user ID: user_20 | Similarity: 0.9257

Top Rekomendasi Lagu:
 1. уйди совсем уйди | Track ID: 6oUDN88DZ8XKIVDfJEyzkN | Skor: 5.0000
 2. faar | Track ID: 4zqoszH8UVaYPx9HMNlt0R | Skor: 5.0000
 3. onks liian myöhä feat nelli petro | Track ID: 1rNwhaO7UD2M08rwKlCNTE | Skor: 5.0000
 4. jby | Track ID: 6L2B0EWlWnHR0YzLCmDgYh | Skor: 5.0000
 5. on lexington 52nd street smash cast version feat will chase | Track ID: 1sFpuKukWsJWI44zfSW2lc | Skor: 5.0000


Rekomendasi untuk User ID: user_2
Top 3 user termirip:
 -user ID: user_15 | Similarity: 0.9280
 -user ID: user_3 | Similarity: 0.9220
 -user ID: user_18 | Similarity: 0.9218

Top Rekomendasi Lagu:
 1. wizz | Track ID: 0Zo3dEV1BOVGF1DaW5aGq9 | Skor: 5.0000
 2. 7 rings | Track ID: 6ocbgoVGwYJhOv1GgI9NsF | Skor: 5.